In [8]:
from collections import defaultdict
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor
from torch_geometric.nn import DistMult
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
from sklearn.manifold import TSNE
from matplotlib import pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'



In [9]:
df = pd.read_csv('../data/edges/clean_triples.csv')

node_keys, node_values = pd.factorize(pd.concat([df['id_entity_1'], df['id_entity_2']]))
rawid2id = {k: v.item() for v, k in zip(node_keys, node_values)}
id2rawid = {v.item(): k  for v, k in zip(node_keys, node_values)}

pred_keys, pred_values = pd.factorize(df['predicate'])
pred2id = {k: v.item() for v, k in zip(pred_keys, pred_values)}
id2pred = {v.item(): k for v, k in zip(pred_keys, pred_values)}

df['id_entity_1'] = df['id_entity_1'].apply(lambda x: rawid2id[x])
df['id_entity_2'] = df['id_entity_2'].apply(lambda x: rawid2id[x])
df['predicate'] = df['predicate'].apply(lambda x: pred2id[x])

hrt_arr = np.array([df['id_entity_1'].to_numpy(), df['predicate'].to_numpy(), df['id_entity_2'].to_numpy()])
hrt_tensor = torch.tensor(hrt_arr, dtype=torch.long).t()



num_triples = hrt_tensor.shape[0]

indices = torch.randperm(num_triples)
train_size = int(0.8 * num_triples)
val_size = int(0.1 * num_triples)

test_indices = indices[train_size + val_size:]
val_indices = indices[train_size : train_size + val_size]
train_indices = indices[:train_size]

train_triplets = hrt_tensor[train_indices].to(device)
val_triplets = hrt_tensor[val_indices].to(device)
test_triplets = hrt_tensor[test_indices].to(device)


filtered_dict = defaultdict(set)

# Обязательно переводим тензор на CPU и конвертируем в список Python
triplets_list = hrt_tensor.cpu().tolist()

for h, r, t in triplets_list:
    filtered_dict[(h, r)].add(t)

In [31]:
pred_keys, pred_values = pd.factorize(df['predicate'])
pred_values

Index([0], dtype='int64')